# Create round_info.csv and Dave config

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd

MERCI_DIR  = Path(os.getcwd()).parent.parent.parent.parent   # MERci/ (notebook lives in MERci/notebooks/prepare_imaging/<variant>/<acquisition>/)
SAMPLE_DIR = MERCI_DIR.parent                  # experiment root, e.g. LT048_sample_18/
sys.path.insert(0, str(MERCI_DIR / "src"))

from MERci.acquisition.dave      import (
    create_round_info, create_round_info_multitissue, create_dave_config,
    create_data_drive_skeleton,
)
from MERci.acquisition.positions import discover_boundary_files
from MERci.acquisition.kilroy    import find_kilroy_config

In [2]:
SETTINGS_DIR  = SAMPLE_DIR / "settings"
METADATA_DIR  = SAMPLE_DIR / "metadata"
POSITIONS_DIR = SAMPLE_DIR / "positions"
METADATA_DIR.mkdir(parents=True, exist_ok=True)

SAMPLE_NAME = SAMPLE_DIR.name
print(f"SAMPLE_DIR   : {SAMPLE_DIR}")
print(f"SAMPLE_NAME  : {SAMPLE_NAME}")

SAMPLE_DIR   : c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time
SAMPLE_NAME  : 251225_LT027_saving_time


In [3]:
# ── Experiment parameters ──────────────────────────────────────────
MICROSCOPE           = "ST2"   # microscope identifier
USE_ADAPTORS         = False    # True = adaptor-based fluidics; False = direct readouts
INCLUDE_FINAL_CLEAVE = True   # True = add a final cleave step after last imaging round
FIRST_HYB_NO_CLEAVE  = True    # True = first hyb (after the cells round) omits the cleave step
DATA_DRIVES          = ["C:", "G:"] #["E", "G:", "F:"]      # e.g. ["D:", "E:", "F:"] to round-robin hyb rounds across physical
                                # drives (cells/transit stay on SAMPLE_DIR's own drive); [] = single-drive

# Default round structure: imaging round 1 = cells only; rounds 2..N_HYBS+1 = bits #1..#N.
# The fluidics before the first bits round has no cleave; later fluidics include the cleave.

# ── Detect the tissue/boundary layout (written by notebook 02) ──────
# >1 boundary -> per-segment recipe (boundary + transit movies); else the classic
# single-positions recipe.
boundaries, MODE = discover_boundary_files(POSITIONS_DIR)
MULTI_BOUNDARY   = len(boundaries) > 1
print(f"Layout mode: {MODE}  ({len(boundaries)} boundary file(s)) -> "
      f"{'per-segment' if MULTI_BOUNDARY else 'single-positions'} recipe")

# HAL config filenames (from notebook 01 / SETTINGS_DIR)
# Adjust these to match the actual files created by notebook 01
bits_hal_configs    = sorted(SETTINGS_DIR.glob("hal-config-*bits*.xml"))
cells_hal_configs   = sorted(SETTINGS_DIR.glob("hal-config-*cells*.xml"))
transit_hal_configs = sorted(SETTINGS_DIR.glob("hal-config-*transit*.xml"))

print("\nAvailable HAL configs in settings/:")
for p in sorted(SETTINGS_DIR.glob("hal-config-*.xml")):
    print(f"  {p.name}")

# Set these manually if auto-detection picks the wrong files
BITS_HAL_CONFIG    = bits_hal_configs[0].name    if bits_hal_configs    else "hal-config-mf3-bits.xml"
CELLS_HAL_CONFIG   = cells_hal_configs[0].name   if cells_hal_configs   else "hal-config-mf3-cells.xml"
TRANSIT_HAL_CONFIG = transit_hal_configs[0].name if transit_hal_configs else None

print(f"\nBits    HAL config : {BITS_HAL_CONFIG}")
print(f"Cells   HAL config : {CELLS_HAL_CONFIG}")
print(f"Transit HAL config : {TRANSIT_HAL_CONFIG}")
print(f"\nUse adaptors        : {USE_ADAPTORS}")
print(f"Final cleave        : {INCLUDE_FINAL_CLEAVE}")
print(f"First hyb no cleave : {FIRST_HYB_NO_CLEAVE}")

if MULTI_BOUNDARY and TRANSIT_HAL_CONFIG is None:
    raise FileNotFoundError(
        "Multiple boundaries detected but no hal-config-*transit*.xml in settings/. "
        "Run the transit cell in notebook 01 first."
    )

Layout mode: single  (2 boundary file(s)) -> per-segment recipe

Available HAL configs in settings/:
  hal-config-st2-bits-blkf15_488f143_560f141_650f141.xml
  hal-config-st2-cells-blkf8_405f141_488f143.xml
  hal-config-st2-transit-blkf2.xml

Bits    HAL config : hal-config-st2-bits-blkf15_488f143_560f141_650f141.xml
Cells   HAL config : hal-config-st2-cells-blkf8_405f141_488f143.xml
Transit HAL config : hal-config-st2-transit-blkf2.xml

Use adaptors        : False
Final cleave        : True
First hyb no cleave : True


## Round – bit – color mapping

Define the round → bit → colour mapping for the codebook. This is the single
source of **`N_HYBS`** (the number of hybridisation/bits rounds, taken as the max
round index) used by the recipe below, and it is saved to `round_bit_color_map.csv`
for notebook 04 to reuse (data organization + Dave annotation).

In [4]:
# round : hyb/bit index (1-indexed), matching the bits movie series number
#         (hal-{mic}-epi_01, _02, …); NOT the Dave imaging-round number.
# bit   : bit number     |     color : excitation wavelength (nm)
round_bit_color = [
    (1,  1,  647), (1,  2,  560),
    (2,  3,  560), (2,  4,  647),
    (3,  5,  647), (3,  6,  560),
    (4,  7,  647), (4,  8,  560),
    (5,  9,  560), (5,  10, 647),
    (6,  11, 647), (6,  12, 560),
    (7,  13, 647), (7,  14, 560),
    (8,  15, 560), (8,  16, 647),
    (9,  17, 647), (9,  18, 560),
    (10, 19, 647), (10, 20, 560),
    (11, 21, 560), (11, 22, 647),
    (12, 23, 647), (12, 24, 560),
    (13, 25, 647), (13, 26, 560),
]

rbc_df   = pd.DataFrame(round_bit_color, columns=["round", "bit", "color"])
rbc_path = METADATA_DIR / "round_bit_color_map.csv"
rbc_df.to_csv(rbc_path, index=False)

N_HYBS = int(rbc_df["round"].max())   # number of bits rounds, derived from the mapping
print(f"Saved: {rbc_path}")
print(f"N_HYBS (from mapping): {N_HYBS}")
print(rbc_df.to_string(index=False))

Saved: c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\metadata\round_bit_color_map.csv
N_HYBS (from mapping): 13
 round  bit  color
     1    1    647
     1    2    560
     2    3    560
     2    4    647
     3    5    647
     3    6    560
     4    7    647
     4    8    560
     5    9    560
     5   10    647
     6   11    647
     6   12    560
     7   13    647
     7   14    560
     8   15    560
     8   16    647
     9   17    647
     9   18    560
    10   19    647
    10   20    560
    11   21    560
    11   22    647
    12   23    647
    12   24    560
    13   25    647
    13   26    560


In [5]:
if DATA_DRIVES:
    create_data_drive_skeleton(
        sample_dir  = SAMPLE_DIR,
        n_bits      = N_HYBS,
        data_drives = DATA_DRIVES,
        mode        = MODE,
        boundaries  = boundaries if MODE == "multi" else None,
    )

if MULTI_BOUNDARY:
    round_info = create_round_info_multitissue(
        microscope         = MICROSCOPE,
        n_bits             = N_HYBS,
        bits_hal_config    = BITS_HAL_CONFIG,
        cells_hal_config   = CELLS_HAL_CONFIG,
        transit_hal_config = TRANSIT_HAL_CONFIG,
        sample_dir         = SAMPLE_DIR,
        boundaries         = boundaries,
        mode               = MODE,
        sample_name        = SAMPLE_NAME,
        data_drives        = DATA_DRIVES or None,
    )
else:
    round_info = create_round_info(
        microscope       = MICROSCOPE,
        n_bits           = N_HYBS,
        bits_hal_config  = BITS_HAL_CONFIG,
        cells_hal_config = CELLS_HAL_CONFIG,
        sample_dir       = SAMPLE_DIR,
        data_drives      = DATA_DRIVES or None,
    )

print(round_info.to_string(index=False))

out_csv = METADATA_DIR / "round_info.csv"
round_info.to_csv(out_csv, index=False)
print(f"\nSaved: {out_csv}")

 imaging_round imaging_type                            series                                             hal_config                                                                                         data_dir                                   positions_file  tissue   segment  fov_start  fov_pad
             1        cells       hal-st2-epi-cells_{fov:03d}         hal-config-st2-cells-blkf8_405f141_488f143.xml   c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data\cells        positions_251225_LT027_saving_time_B1.txt       1        B1          0        3
             1      transit hal-st2-epi-transit_r01_{fov:03d}                       hal-config-st2-transit-blkf2.xml c:\Users\Leonardo\Dropbox\research\analysis\LineageTracing\251225_LT027_saving_time\data\transit positions_251225_LT027_saving_time_transit_1.txt       1 transit_1          0        3
             1        cells       hal-st2-epi-cells_{fov:03d}         hal-config-st2-cells-blkf8_

In [6]:
# ── Resolve positions inputs for the recipe ──────────────────────────────
if MULTI_BOUNDARY:
    # Per-segment: each round_info row names its own positions file in POSITIONS_DIR.
    positions_arg     = None
    positions_dir_arg = POSITIONS_DIR
    missing = [f for f in round_info["positions_file"].unique()
               if not (POSITIONS_DIR / f).exists()]
    if missing:
        raise FileNotFoundError(
            f"Positions files referenced by round_info are missing (run notebook 02): {missing}"
        )
else:
    # Single-positions: one file for every movie.
    positions_arg     = POSITIONS_DIR / f"positions_{SAMPLE_NAME}.txt"
    positions_dir_arg = None
    if not positions_arg.exists():
        raise FileNotFoundError(f"Positions file not found: {positions_arg}")

# Resolve the Kilroy config that will run this experiment. Its protocol names are
# the source of truth for the fluidic steps written into the Dave recipe, so every
# protocol referenced is guaranteed to exist in Kilroy. If the microscope has no
# Kilroy config, fall back to MF2's.
KILROY_DIR    = MERCI_DIR / "data" / "configs" / "kilroy"
KILROY_CONFIG = find_kilroy_config(MICROSCOPE, KILROY_DIR, fallback_microscope="MF2")
print(f"Kilroy config (protocol source): {KILROY_CONFIG.name}")

dave_name   = f"dave-{MICROSCOPE.lower()}-{N_HYBS}hybs-{SAMPLE_NAME}.xml"
dave_output = SETTINGS_DIR / dave_name

create_dave_config(
    round_info           = round_info,
    positions_file       = positions_arg,
    settings_dir         = SETTINGS_DIR,
    output_path          = dave_output,
    use_adaptors         = USE_ADAPTORS,
    include_final_cleave = INCLUDE_FINAL_CLEAVE,
    first_hyb_no_cleave  = FIRST_HYB_NO_CLEAVE,
    kilroy_config        = KILROY_CONFIG,
    positions_dir        = positions_dir_arg,
)

print(f"Dave config saved: {dave_output}")

# Preview the generated file
with open(dave_output, encoding="ISO-8859-1") as fh:
    print(fh.read())

Kilroy config (protocol source): kilroy-config-st2-240528.xml


ValueError: Ambiguous Kilroy protocol for fluidic step 'cleave (direct)': ['Cleave', 'Cleave Slow', 'Cleave then image']. Cannot decide which to use.

## Kilroy config

Locates the Kilroy config for the current microscope in `MERci/data/configs/kilroy/`
(newest matching file by `YYMMDD` date stamp) and copies it to `SAMPLE_DIR/settings/`.
This is the same file used above as the protocol source for the Dave recipe. If the
microscope has no Kilroy config, it falls back to MF2's.

In [ ]:
import shutil

# Same resolution (with MF2 fallback) used as the Dave protocol source above.
kilroy_src  = find_kilroy_config(MICROSCOPE, MERCI_DIR / "data" / "configs" / "kilroy",
                                 fallback_microscope="MF2")
kilroy_dest = SETTINGS_DIR / kilroy_src.name
shutil.copy2(str(kilroy_src), str(kilroy_dest))
print(f"Kilroy config  : {kilroy_src.name}")
print(f"Copied to      : {kilroy_dest}")